# Create VGGFace2 Landmarks With FAN

This notebook uses the FAN wrapper from `/home/jie/Documents/NextFace_custom/landmarks/fan.py` to create DECA-style landmark `.npy` files for raw VGGFace2 images.

Input image layout:

```text
/home/jie/Downloads/vggface2_train/train/<identity>/<image>.jpg
```

Output landmark layout:

```text
/home/jie/Downloads/vggface2_train/train_annotated_fan/<identity>/<image>.npy
```

That output folder can later be used as DECA's `kptfolder`. Each `.npy` contains FAN 68-point 2D landmarks with shape `[68, 2]` in original image pixel coordinates.

In [1]:
from pathlib import Path
import sys

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
from skimage.io import imread
from tqdm.auto import tqdm

PROJECT_ROOT = Path('/home/jie/Documents/NextFace_custom')
DECA_ROOT = PROJECT_ROOT / 'ext' / 'deca'
sys.path.insert(0, str(PROJECT_ROOT))

IMAGE_ROOT = Path('/home/jie/Downloads/vggface2_train/train')
FIRST_PASS_KPT_ROOT = Path('/home/jie/Downloads/vggface2_train/train_annotated_fan')
KPT_ROOT = FIRST_PASS_KPT_ROOT

CLEAN_LIST_PATH = Path('/home/jie/Downloads/vggface2_train/vggface2_train_fan_stability_clean_list.npy')
CLEAN_METADATA_PATH = Path('/home/jie/Downloads/vggface2_train/vggface2_train_fan_stability_clean_metadata.npz')
ACCEPTED_NAMES_PATH = Path('/home/jie/Downloads/vggface2_train/vggface2_train_fan_stability_clean_names.txt')

FIRST_PASS_METADATA_PATH = FIRST_PASS_KPT_ROOT.parent / 'fan_landmark_creation_metadata.npz'
FIRST_PASS_SAVED_NAMES_PATH = FIRST_PASS_KPT_ROOT.parent / 'fan_landmark_creation_saved_names.txt'
FIRST_PASS_FAILURE_LOG_PATH = FIRST_PASS_KPT_ROOT.parent / 'fan_landmark_failures.txt'

print('PROJECT_ROOT =', PROJECT_ROOT)
print('IMAGE_ROOT =', IMAGE_ROOT, IMAGE_ROOT.exists())
print('FIRST_PASS_KPT_ROOT =', FIRST_PASS_KPT_ROOT, FIRST_PASS_KPT_ROOT.exists())
print('CLEAN_LIST_PATH =', CLEAN_LIST_PATH)
print('FIRST_PASS_METADATA_PATH =', FIRST_PASS_METADATA_PATH, FIRST_PASS_METADATA_PATH.exists())


PROJECT_ROOT = /home/jie/Documents/NextFace_custom
IMAGE_ROOT = /home/jie/Downloads/vggface2_train/train True
FIRST_PASS_KPT_ROOT = /home/jie/Downloads/vggface2_train/train_annotated_fan True
CLEAN_LIST_PATH = /home/jie/Downloads/vggface2_train/vggface2_train_fan_stability_clean_list.npy
FIRST_PASS_METADATA_PATH = /home/jie/Downloads/vggface2_train/fan_landmark_creation_metadata.npz True


/home/jie/Documents/NextFace_custom/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load FAN

`LandmarksDetectorFAN` takes a mask of landmark indices. For DECA/VGGFace2, use all 68 FAN landmarks, so the mask is `torch.arange(68)`.

The first run may take longer because the `face_alignment` package may initialize model weights.

In [2]:
from landmarks.fan import LandmarksDetectorFAN

device = 'cuda' if torch.cuda.is_available() else 'cpu'
mask = torch.arange(68, dtype=torch.long)
fan = LandmarksDetectorFAN(mask=mask, device=device)

print('device =', device)

device = cuda


## Collect Images

The output path is computed by preserving the path relative to `IMAGE_ROOT` and changing `.jpg` to `.npy`.

In [3]:
image_paths = sorted(
    list(IMAGE_ROOT.glob('*/*.jpg'))
    + list(IMAGE_ROOT.glob('*/*.jpeg'))
    + list(IMAGE_ROOT.glob('*/*.png'))
)
image_path_by_name = {
    image_path.relative_to(IMAGE_ROOT).with_suffix('').as_posix(): image_path
    for image_path in image_paths
}

print('num images:', len(image_paths))
print('first image:', image_paths[0] if image_paths else None)


def image_name_for_path(image_path):
    return Path(image_path).relative_to(IMAGE_ROOT).with_suffix('').as_posix()


def kpt_path_for_image(image_path):
    rel = Path(image_path).relative_to(IMAGE_ROOT)
    return (FIRST_PASS_KPT_ROOT / rel).with_suffix('.npy')


if image_paths:
    print('first cached first-pass landmarks:', kpt_path_for_image(image_paths[0]))


num images: 3141890
first image: /home/jie/Downloads/vggface2_train/train/n000002/0001_01.jpg
first cached first-pass landmarks: /home/jie/Downloads/vggface2_train/train_annotated_fan/n000002/0001_01.npy


## Count First-Pass Landmark Progress

`fan_landmark_creation_metadata.npz` is the source of truth for which images have already been scanned. The annotated folder is still counted as a consistency check, but the next candidate is chosen from metadata progress.

In [4]:
LIMIT = 1000  # keep None to count all unscanned first-pass candidates

if not FIRST_PASS_METADATA_PATH.is_file():
    raise FileNotFoundError(FIRST_PASS_METADATA_PATH)

def metadata_get(metadata, key, default):
    return metadata[key] if key in metadata.files else default

all_image_names = list(image_path_by_name.keys())
all_image_name_set = set(all_image_names)

metadata = np.load(FIRST_PASS_METADATA_PATH, allow_pickle=False)
saved_names = metadata_get(metadata, 'saved_names', np.asarray([], dtype=str)).astype(str)
failure_names = metadata_get(metadata, 'failure_names', np.asarray([], dtype=str)).astype(str)
failure_errors = metadata_get(metadata, 'failure_errors', np.full(len(failure_names), '', dtype=str)).astype(str)

saved_name_set = set(saved_names.tolist())
failure_name_set = set(failure_names.tolist())
scanned_name_set = saved_name_set | failure_name_set

first_pass_paths = sorted(FIRST_PASS_KPT_ROOT.glob('*/*.npy'))
annotated_names = {
    kpt_path.relative_to(FIRST_PASS_KPT_ROOT).with_suffix('').as_posix()
    for kpt_path in first_pass_paths
}
matching_annotated_names = annotated_names & all_image_name_set
orphan_annotated_names = annotated_names - all_image_name_set

saved_with_source_names = saved_name_set & all_image_name_set
failure_with_source_names = failure_name_set & all_image_name_set
scanned_with_source_names = scanned_name_set & all_image_name_set
metadata_names_without_source = scanned_name_set - all_image_name_set
annotated_not_in_metadata = matching_annotated_names - saved_name_set
metadata_saved_missing_file = saved_with_source_names - annotated_names

unscanned_names = [name for name in all_image_names if name not in scanned_name_set]

if LIMIT is None:
    names_to_scan = unscanned_names
elif LIMIT < 0:
    raise ValueError('LIMIT must be None or a non-negative integer')
else:
    names_to_scan = unscanned_names[:LIMIT]

print('image root:', IMAGE_ROOT)
print('annotated folder:', FIRST_PASS_KPT_ROOT)
print('metadata path:', FIRST_PASS_METADATA_PATH)
print('total source images:', len(all_image_names))
print('quantity of metadata saved:', len(saved_names))


image root: /home/jie/Downloads/vggface2_train/train
annotated folder: /home/jie/Downloads/vggface2_train/train_annotated_fan
metadata path: /home/jie/Downloads/vggface2_train/fan_landmark_creation_metadata.npz
total source images: 3141890
quantity of metadata saved: 411773


In [6]:
# Run metadata-based candidates selected by LIMIT through FAN, save .npy files, and keep results in memory.
def read_image_rgb_uint8(image_path):
    image = imread(image_path)
    if image.ndim == 2:
        image = np.repeat(image[..., None], 3, axis=2)
    if image.shape[2] == 4:
        image = image[..., :3]
    if image.dtype == np.uint8:
        return image
    return np.clip(image, 0, 255).astype(np.uint8)


if 'names_to_scan' not in globals():
    if 'unscanned_names' not in globals():
        if not FIRST_PASS_METADATA_PATH.is_file():
            raise FileNotFoundError(FIRST_PASS_METADATA_PATH)
        all_image_names = list(image_path_by_name.keys())
        metadata = np.load(FIRST_PASS_METADATA_PATH, allow_pickle=False)
        saved_names = metadata['saved_names'].astype(str) if 'saved_names' in metadata.files else np.asarray([], dtype=str)
        failure_names = metadata['failure_names'].astype(str) if 'failure_names' in metadata.files else np.asarray([], dtype=str)
        scanned_name_set = set(saved_names.tolist()) | set(failure_names.tolist())
        unscanned_names = [name for name in all_image_names if name not in scanned_name_set]
    if 'LIMIT' not in globals():
        LIMIT = 1000
    names_to_scan = unscanned_names if LIMIT is None else unscanned_names[:LIMIT]

fan_landmark_results = []
fan_landmark_failures = []
mask_np = fan.mask

for name in tqdm(names_to_scan, desc='running FAN'):
    image_path = image_path_by_name[name]
    try:
        image = read_image_rgb_uint8(image_path)
        with torch.no_grad():
            detected = fan.landmarksDetector.get_landmarks_from_image(image, None)
        if detected is None or len(detected) == 0:
            raise RuntimeError(f'No landmarks found for {name}.jpg')

        landmarks = np.asarray(detected[0][mask_np][:, :2], dtype=np.float32)
        output_path = kpt_path_for_image(image_path)
        output_path.parent.mkdir(parents=True, exist_ok=True)
        np.save(output_path, landmarks)
        fan_landmark_results.append({
            'name': name,
            'image_path': image_path,
            'output_path': output_path,
            'landmarks': landmarks,
        })
    except Exception as exc:
        fan_landmark_failures.append((name, repr(exc)))

print('attempted landmark detections:', len(names_to_scan))
print('successful landmark detections:', len(fan_landmark_results))
print('failed landmark detections:', len(fan_landmark_failures))
print('first failures:', fan_landmark_failures[:5])
fan_landmark_results[:5]


running FAN:   6%|▌         | 61/1000 [00:22<05:43,  2.73it/s]


KeyboardInterrupt: 

In [7]:
# Save the current FAN run results into fan_landmark_creation_metadata.npz.
if 'fan_landmark_results' not in globals():
    raise RuntimeError('Run the tqdm FAN cell before updating metadata.')
if 'fan_landmark_failures' not in globals():
    fan_landmark_failures = []
if not FIRST_PASS_METADATA_PATH.is_file():
    raise FileNotFoundError(FIRST_PASS_METADATA_PATH)

metadata = np.load(FIRST_PASS_METADATA_PATH, allow_pickle=False)
saved_names = metadata['saved_names'].astype(str).tolist() if 'saved_names' in metadata.files else []
saved_existing_flags = metadata['saved_existing_flags'].astype(bool).tolist() if 'saved_existing_flags' in metadata.files else [False] * len(saved_names)
failure_names = metadata['failure_names'].astype(str).tolist() if 'failure_names' in metadata.files else []
failure_errors = metadata['failure_errors'].astype(str).tolist() if 'failure_errors' in metadata.files else [''] * len(failure_names)

saved_existing_by_name = {
    name: bool(saved_existing_flags[i]) if i < len(saved_existing_flags) else False
    for i, name in enumerate(saved_names)
}
failure_errors_by_name = {
    name: str(failure_errors[i]) if i < len(failure_errors) else ''
    for i, name in enumerate(failure_names)
}

saved_added = 0
for record in fan_landmark_results:
    name = str(record['name'])
    if name not in saved_existing_by_name:
        saved_added += 1
    saved_existing_by_name[name] = False
    failure_errors_by_name.pop(name, None)

failures_added_or_updated = 0
for name, error in fan_landmark_failures:
    name = str(name)
    if name not in saved_existing_by_name:
        if failure_errors_by_name.get(name) != str(error):
            failures_added_or_updated += 1
        failure_errors_by_name[name] = str(error)

updated_saved_names = np.asarray(list(saved_existing_by_name.keys()), dtype=str)
updated_saved_existing_flags = np.asarray(
    [saved_existing_by_name[name] for name in saved_existing_by_name],
    dtype=bool,
)
updated_failure_names = np.asarray(list(failure_errors_by_name.keys()), dtype=str)
updated_failure_errors = np.asarray(
    [failure_errors_by_name[name] for name in failure_errors_by_name],
    dtype=str,
)

np.savez_compressed(
    FIRST_PASS_METADATA_PATH,
    saved_names=updated_saved_names,
    saved_existing_flags=updated_saved_existing_flags,
    failure_names=updated_failure_names,
    failure_errors=updated_failure_errors,
)

print('metadata updated:', FIRST_PASS_METADATA_PATH)
print('saved names before:', len(saved_names))
print('saved names after:', len(updated_saved_names))
print('new saved names added:', saved_added)
print('failure names before:', len(failure_names))
print('failure names after:', len(updated_failure_names))
print('failures added or updated:', failures_added_or_updated)


metadata updated: /home/jie/Downloads/vggface2_train/fan_landmark_creation_metadata.npz
saved names before: 411773
saved names after: 411773
new saved names added: 0
failure names before: 54
failure names after: 115
failures added or updated: 61
